# 01 — Interface bottleneck

Analysis 6.1: isolate **compression/subspace selection** from **endpoint orientation**. The notebook runs the controlled interface grid and exports one compact projection/interface ablation table. The former multi-panel PCA figure is intentionally removed: dimensionality, geometry retention, and downstream transfer now stay together in the table.

Training and analysis are separate: set `EXECUTE=True` only when launching missing runs; completed runs are resumed by their final-test record.


In [ ]:
# 1. Cấu hình thí nghiệm
from pathlib import Path

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
AUTO_PULL_REPO = True
INSTALL_REQUIREMENTS = True

PAIR = "qwen3_4b_to_bert_base"
TRAIN_DATA_REL = Path("data/train_set/train_100k.csv")
RUN_NAME = f"analysis_interface_{PAIR}_v1"
SEEDS = [42, 43, 44]
RANDOM_DRAWS = [0, 1, 2]
BATCH_SIZE, EPOCHS, LR = 128, 5, 7e-5
STUDENT_DIM = 768
DIMS = [64, 128, 256, 384, STUDENT_DIM]
GEOMETRY_ROWS = 2048
EXECUTE = False
EVAL_RETRIEVAL = False
CUDA_VISIBLE_DEVICES = "0"


In [ ]:
# 2. Clone/fetch repo, dependencies, imports và output
import shlex, subprocess, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_DIR = next((p for p in (cwd, cwd.parent) if (p / "main.py").is_file()), None)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"
if AUTO_PULL_REPO:
    dirty = subprocess.run(["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"], check=True, capture_output=True, text=True).stdout.strip()
    if dirty:
        print("[git] Bỏ qua pull vì repo có tracked changes.")
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
git_head = subprocess.run(["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / "notebooks"))
from _analysis_common import PAIRS, collect_jobs, geoode_command, load_teacher_cache, run_jobs, set_paper_style, teacher_cache_path
from src import structural_audit as audit
from src.teacher_projection import retained_energy

PAIR_CONFIG = PAIRS[PAIR]
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"
OUTPUT_BASE = PROJECT_DIR / "runs"
RUN_ROOT = OUTPUT_BASE / RUN_NAME
CACHE_DIR = OUTPUT_BASE / "teacher_cache"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
set_paper_style()
print(f"Repo: {PROJECT_DIR} @ {git_head}")
print(f"{PAIR}: {PAIR_CONFIG['teacher']} -> {PAIR_CONFIG['student']}")
print(f"Output: {RUN_ROOT}")


In [ ]:
# 3. Một job cho mỗi (interface, draw, training seed)
ARM_SPECS = [
    {"arm": "learned_t2s", "draws": [None], "refit": 0, "args": ["--projection_type", "learned_t2s", "--no-gauge_align"]},
    {"arm": "random_proc", "draws": RANDOM_DRAWS, "refit": 1, "args": ["--projection_type", "random", "--gauge_align", "--gauge_rotation", "procrustes"]},
    {"arm": "pca_none", "draws": [None], "refit": 0, "args": ["--projection_type", "pca", "--no-gauge_align"]},
    {"arm": "pca_proc", "draws": [None], "refit": 1, "args": ["--projection_type", "pca", "--gauge_align", "--gauge_rotation", "procrustes"]},
]
jobs = []
for spec in ARM_SPECS:
    for draw in spec["draws"]:
        for seed in SEEDS:
            suffix = spec["arm"] if draw is None else f"{spec['arm']}__d{draw}"
            run_dir = RUN_ROOT / suffix / f"seed_{seed}"
            extra = ["--lambda_end", "1", "--lambda_ctr", "0", "--lambda_topo", "0", "--gauge_refit_every", str(spec["refit"]), *spec["args"]]
            if draw is not None:
                extra += ["--projection_seed", str(draw)]
            if not EVAL_RETRIEVAL:
                extra += ["--no_eval_retrieval"]
            jobs.append({
                "name": f"{suffix}/seed_{seed}", "arm": spec["arm"], "draw": draw, "seed": seed, "run_dir": run_dir,
                "command": geoode_command(PROJECT_DIR, pair=PAIR_CONFIG, train_data=TRAIN_DATA, cache_dir=CACHE_DIR, run_dir=run_dir, seed=seed, batch_size=BATCH_SIZE, epochs=EPOCHS, learning_rate=LR, extra=extra),
            })
print(f"Plan: {len(jobs)} jobs")
for job in jobs:
    print(shlex.join(job["command"]))
if EXECUTE:
    display(run_jobs(PROJECT_DIR, jobs, cuda_visible_devices=CUDA_VISIBLE_DEVICES))


In [ ]:
# 4. Aggregate interface results
results = collect_jobs(jobs)
results.to_csv(RUN_ROOT / "interface_results.csv", index=False)
done = results.query("status == 'done'").copy()
if done.empty:
    print("No completed runs yet. Set EXECUTE=True or point RUN_NAME to an existing grid.")
else:
    summary = done.groupby("arm").agg(avg_all=("avg_all", "mean"), std=("avg_all", "std"), n=("avg_all", "count"), energy=("explained_energy", "mean"), init_cos=("cos_after", "mean")).reset_index()
    display(summary.style.format({"avg_all": "{:.4f}", "std": "{:.4f}", "energy": "{:.3f}", "init_cos": "{:+.3f}"}))


In [ ]:
# 5. Table — projection/interface ablation (replaces the former PCA figure)
cache_path = teacher_cache_path(PROJECT_DIR, CACHE_DIR, pair=PAIR_CONFIG, train_data=TRAIN_DATA)
teacher, cache_meta = load_teacher_cache(cache_path)
rng = np.random.default_rng(0)
keep = np.sort(rng.choice(len(teacher), size=min(GEOMETRY_ROWS, len(teacher)), replace=False))
teacher = teacher[torch.as_tensor(keep)].float()
target_dim = min(STUDENT_DIM, teacher.shape[1])
geometry_rows = []
for kind, draws in (("pca", [0]), ("random", RANDOM_DRAWS)):
    for draw in draws:
        projection, mean = audit.fit_variant(teacher, kind, target_dim, seed=draw)
        target = audit.apply_map(teacher, projection, mean=mean, subtract_mean=False)
        geometry_rows.append({
            "geometry_kind": kind, "draw": draw, "dimension": target_dim,
            "retained_energy": retained_energy(teacher, projection),
            "gram_rmse": audit.gram_rmse(target, teacher),
            "knn_overlap@10": audit.knn_overlap(target, teacher, k=10),
        })
geometry = pd.DataFrame(geometry_rows)
geometry.to_csv(RUN_ROOT / "projection_geometry_by_draw.csv", index=False)
geometry_summary = geometry.groupby("geometry_kind").agg(
    dimension=("dimension", "first"), retained_energy=("retained_energy", "mean"),
    gram_rmse=("gram_rmse", "mean"), knn_overlap_at_10=("knn_overlap@10", "mean")
).reset_index()
interface = done.groupby("arm").agg(final_score=("avg_all", "mean"), score_sd=("avg_all", "std"), n=("avg_all", "count")).reset_index()
labels = {"learned_t2s": "Learned projector", "random_proc": "Random projection + Procrustes", "pca_none": "PCA", "pca_proc": "PCA + epoch-wise Procrustes"}
geometry_kind = {"random_proc": "random", "pca_none": "pca", "pca_proc": "pca"}
interface["variant"] = interface["arm"].map(labels)
interface["geometry_kind"] = interface["arm"].map(geometry_kind)
table = interface.merge(geometry_summary, on="geometry_kind", how="left").drop(columns=["geometry_kind"])
table = table[["variant", "dimension", "retained_energy", "gram_rmse", "knn_overlap_at_10", "final_score", "score_sd", "n"]]
table.to_csv(RUN_ROOT / "table_projection_interface.csv", index=False)
display(table.style.format({"dimension": "{:.0f}", "retained_energy": "{:.3f}", "gram_rmse": "{:.4f}", "knn_overlap_at_10": "{:.3f}", "final_score": "{:.4f}", "score_sd": "{:.4f}"}, na_rep="—"))
